In [2]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('/content/results_fixed.csv', sep=',', encoding='utf-8', na_values=['', ' '])
def normalize_cell(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "n/a", "na", "-", "—"}:
        return np.nan
    return s
clean = df.apply(lambda col: col.map(normalize_cell))
filled_count = clean.notna().sum(axis=1)
rows_to_keep = ~(
    (filled_count == 0) |
    ((filled_count == 1) & clean["request_key"].notna())
)
df = df.loc[rows_to_keep].copy()

df.head()

,request_key,категория,запрос,тип_запроса,домены_источников,тип_источника,есть_ли_pg,конкуренты,кто_выигрывает,пробелы
0,herbal essences алоэ,hair care,herbal essences алоэ,брендовый,"poryadok.ru, ozon.ru, global.podrygka.ru, gold...",маркетплейс,Да,NaN,P&G,NaN
1,oral b,oral care,oral b,брендовый,"oralb-russia.ru, market.yandex.ru, irrigator.r...",перекуп,Да,NaN,P&G,Нет объяснения критериев выбора
2,подгузники для 7 летних девочек,baby care,подгузники для 7 летних девочек,категорийный,"wildberries.ru, market.yandex.ru, ozon.ru, vk....",маркетплейс,Да,"Huggies, Helen Harper",конкуренты,"P&G есть, но не в лидирующей рекомендации, Нет..."
3,как выбрать шампунь для окрашенных,hair care,как выбрать шампунь для окрашенных,проблемно-решающий,"letu.ru, esteticshop.ru, lady.mail.ru, dzen.ru...",обзор,Нет,NaN,конкуренты,"Нет присутствия P&G в AI-ответе, Конкурент выи..."
4,тена подгузники размеры таблица,baby care,тена подгузники размеры таблица,проблемно-решающий,"comfer.ru, vitaexpress.ru, apteka.ru, uteka.ru...",перекуп,Нет,Tena,конкуренты,"Нет присутствия P&G в AI-ответе, Конкурент выи..."


In [8]:
domains = df[["запрос", "категория", "тип_запроса", "домены_источников"]].copy()
domains["домены_источников"] = domains["домены_источников"].fillna("").str.split(",")
domains = domains.explode("домены_источников")
domains["домены_источников"] = domains["домены_источников"].str.strip().str.lower()
domains = domains[domains["домены_источников"] != ""]
domains

,запрос,категория,тип_запроса,домены_источников
0,herbal essences алоэ,hair care,брендовый,poryadok.ru
0,herbal essences алоэ,hair care,брендовый,ozon.ru
0,herbal essences алоэ,hair care,брендовый,global.podrygka.ru
0,herbal essences алоэ,hair care,брендовый,goldapple.ru
0,herbal essences алоэ,hair care,брендовый,online.globus.ru
...,...,...,...,...
691,маска бальзам шампунь для волос,hair care,категорийный,letu.ru
691,маска бальзам шампунь для волос,hair care,категорийный,ozon.ru
691,маска бальзам шампунь для волос,hair care,категорийный,biofollica.ru
691,маска бальзам шампунь для волос,hair care,категорийный,wildberries.ru


In [9]:
competitors = df[["запрос", "категория", "тип_запроса", "конкуренты"]].copy()
competitors["конкуренты"] = competitors["конкуренты"].fillna("").str.split(",")
competitors = competitors.explode("конкуренты")
competitors["конкуренты"] = competitors["конкуренты"].str.strip().str.lower()
competitors = competitors[competitors["конкуренты"] != ""]
competitors

,запрос,категория,тип_запроса,конкуренты
2,подгузники для 7 летних девочек,baby care,категорийный,huggies
2,подгузники для 7 летних девочек,baby care,категорийный,helen harper
4,тена подгузники размеры таблица,baby care,проблемно-решающий,tena
7,себозол шампунь от перхоти,hair care,категорийный,себозол
9,шампунь ополаскиватель 2 в 1,ОШИБКА,ОШИБКА,ошибка
...,...,...,...,...
683,паста для снижения чувствительности зубов рейтинг,oral care,категорийный,innova
686,пантин для волос,ОШИБКА,ОШИБКА,ошибка
691,маска бальзам шампунь для волос,hair care,категорийный,royal samples
691,маска бальзам шампунь для волос,hair care,категорийный,oil premium


In [10]:
pg_stats = (
    df["есть_ли_pg"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"nan": None, "": None})
)

pg_stats = (
    pg_stats[pg_stats.isin(["да", "нет"])]
    .value_counts()
    .rename_axis("есть_ли_pg")
    .reset_index(name="count")
)

pg_stats["proportion"] = pg_stats["count"] / pg_stats["count"].sum()
pg_stats["proportion_pct"] = (pg_stats["proportion"] * 100).round(1).astype(str) + "%"

pg_stats


,есть_ли_pg,count,proportion,proportion_pct
0,нет,404,0.644338,64.4%
1,да,223,0.355662,35.6%


In [11]:
df["кто_выигрывает"].value_counts(normalize=True) * 100

,proportion
кто_выигрывает,
конкуренты,63.150289
P&G,26.445087
ОШИБКА,9.393064
одинаково,1.011561


In [12]:
min_pct = 1

domain_share = (
    domains.groupby("домены_источников")["запрос"]
    .nunique()
    .div(df["запрос"].nunique())
    .mul(100)
    .sort_values(ascending=False)
)

domain_share = domain_share[domain_share >= min_pct]

domain_share.map(lambda x: f"{x:.2f}%")
# если процент запросов в которых встречается домен слииишком мал (меньше 1) - не учитываб

,запрос
домены_источников,
market.yandex.ru,37.86%
ozon.ru,32.08%
apteka.ru,18.50%
wildberries.ru,17.20%
goldapple.ru,15.03%
...,...
luberdent.ru,1.01%
medaboutme.ru,1.01%
r-ulybka.ru,1.01%


In [13]:
df["тип_источника"].value_counts(normalize=True) * 100

,proportion
тип_источника,
обзор,27.062229
маркетплейс,23.733719
перекуп,21.273517
экспертный ресурс,12.879884
ОШИБКА,9.406657
сайт бренда,3.762663
форум,1.881331


In [14]:
pd.crosstab(df["тип_запроса"], df["есть_ли_pg"], normalize="index") * 100

есть_ли_pg,Да,Нет,ОШИБКА
тип_запроса,,,
ОШИБКА,0.000000,0.000000,100.0
брендовый,81.067961,18.932039,0.0
категорийный,17.511521,82.488479,0.0
проблемно-решающий,8.823529,91.176471,0.0


In [15]:
df["пробелы_clean"] = (
    df["пробелы"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
df["пробелы_clean"]

,пробелы_clean
0,
1,нет объяснения критериев выбора
2,"p&g есть, но не в лидирующей рекомендации, нет..."
3,"нет присутствия p&g в ai-ответе, конкурент выи..."
4,"нет присутствия p&g в ai-ответе, конкурент выи..."
...,...
687,
688,"p&g есть, но не в лидирующей рекомендации, нет..."
689,"нет присутствия p&g в ai-ответе, конкурент выи..."
690,"нет присутствия p&g в ai-ответе, конкурент выи..."


In [16]:
df["пробелы_clean"] = (
    df["пробелы"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)
# GAP_PATTERNS = {
#    "no_pg_in_ai": r"нет присутствия p&g в ai-ответе",
#    "pg_not_leading": r"p&g есть,\s*но не в лидирующей рекомендации",
#    "no_category_content": r"присутствует только в брендовых запросах,\s*но отсутствует в категорийных",
#    "no_consultation_content": r"присутствует в категорийных,\s*но отсутствует в консультационных",
#    "no_comparison_content": r"нет контента под сравнение с конкурентами",
#    "no_selection_criteria": r"нет объяснения критериев выбора",
#    "weak_authority_sources": r"недостаточно авторитетных внешних источников",
#    "weak_purchase_platform_presence": r"слабое присутствие на площадках,\s*влияющих на покупку",
#    "competitor_problem_solution_better": r"у конкурента лучше покрыт problem-solution сценарий",
#    "competitor_comparison_better": r"у конкурента лучше покрыт comparison-сценарий",
#}

GAP_PATTERNS = {
    "no_pg_in_ai": r"нет присутствия p&g в ai-ответе",
    "pg_not_leading": r"p&g есть,\s*но не в лидирующей рекомендации",
    "no_selection_criteria": r"нет объяснения критериев выбора",
    "competitor_problem_solution_better": r"у конкурента лучше покрыт problem-solution сценарий",
    "competitor_comparison_better": r"у конкурента лучше покрыт comparison-сценарий",
}

for gap_name, pattern in GAP_PATTERNS.items():
    df[gap_name] = df["пробелы_clean"].str.contains(pattern, regex=True, na=False).astype(int)

gap_cols = list(GAP_PATTERNS.keys())

df["has_any_gap"] = (df[gap_cols].sum(axis=1) > 0).astype(int)
df["gap_count"] = df[gap_cols].sum(axis=1)

df.head()



,request_key,категория,запрос,тип_запроса,домены_источников,тип_источника,есть_ли_pg,конкуренты,кто_выигрывает,пробелы,пробелы_clean,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better,has_any_gap,gap_count
0,herbal essences алоэ,hair care,herbal essences алоэ,брендовый,"poryadok.ru, ozon.ru, global.podrygka.ru, gold...",маркетплейс,Да,NaN,P&G,NaN,,0,0,0,0,0,0,0
1,oral b,oral care,oral b,брендовый,"oralb-russia.ru, market.yandex.ru, irrigator.r...",перекуп,Да,NaN,P&G,Нет объяснения критериев выбора,нет объяснения критериев выбора,0,0,1,0,0,1,1
2,подгузники для 7 летних девочек,baby care,подгузники для 7 летних девочек,категорийный,"wildberries.ru, market.yandex.ru, ozon.ru, vk....",маркетплейс,Да,"Huggies, Helen Harper",конкуренты,"P&G есть, но не в лидирующей рекомендации, Нет...","p&g есть, но не в лидирующей рекомендации, нет...",0,1,1,0,0,1,2
3,как выбрать шампунь для окрашенных,hair care,как выбрать шампунь для окрашенных,проблемно-решающий,"letu.ru, esteticshop.ru, lady.mail.ru, dzen.ru...",обзор,Нет,NaN,конкуренты,"Нет присутствия P&G в AI-ответе, Конкурент выи...","нет присутствия p&g в ai-ответе, конкурент выи...",1,0,0,0,0,1,1
4,тена подгузники размеры таблица,baby care,тена подгузники размеры таблица,проблемно-решающий,"comfer.ru, vitaexpress.ru, apteka.ru, uteka.ru...",перекуп,Нет,Tena,конкуренты,"Нет присутствия P&G в AI-ответе, Конкурент выи...","нет присутствия p&g в ai-ответе, конкурент выи...",1,0,0,0,0,1,1


In [17]:
GAP_LABELS = {
    "no_pg_in_ai": "Нет присутствия P&G в AI-ответе",
    "pg_not_leading": "P&G есть, но не в лидирующей рекомендации",
    "no_selection_criteria": "Нет объяснения критериев выбора",
    "competitor_problem_solution_better": "Конкурент сильнее в problem-solution",
    "competitor_comparison_better": "Конкурент сильнее в comparison",
}

#GAP_LABELS = {
#    "no_pg_in_ai": "Нет присутствия P&G в AI-ответе",
#    "pg_not_leading": "P&G есть, но не в лидирующей рекомендации",
#    "no_category_content": "Нет категорийного покрытия",
#    "no_consultation_content": "Нет консультационного покрытия",
#    "no_comparison_content": "Нет comparison-контента",
#    "no_selection_criteria": "Нет объяснения критериев выбора",
#    "weak_authority_sources": "Недостаточно авторитетных внешних источников",
#   "weak_purchase_platform_presence": "Слабое присутствие на purchase-площадках",
#   "competitor_problem_solution_better": "Конкурент сильнее в problem-solution",
#   "competitor_comparison_better": "Конкурент сильнее в comparison",
#}


gap_summary = (
    df[gap_cols]
    .sum()
    .rename("count")
    .reset_index()
    .rename(columns={"index": "gap"})
)

gap_summary["proportion"] = gap_summary["count"] / len(df)
gap_summary["proportion_pct"] = (gap_summary["proportion"] * 100).round(1)
gap_summary = gap_summary.sort_values("count", ascending=False)
gap_summary["gap_label"] = gap_summary["gap"].map(GAP_LABELS)
gap_summary = gap_summary[["gap", "gap_label", "count", "proportion", "proportion_pct"]]
gap_summary

,gap,gap_label,count,proportion,proportion_pct
0,no_pg_in_ai,Нет присутствия P&G в AI-ответе,402,0.580925,58.1
2,no_selection_criteria,Нет объяснения критериев выбора,149,0.215318,21.5
4,competitor_comparison_better,Конкурент сильнее в comparison,60,0.086705,8.7
1,pg_not_leading,"P&G есть, но не в лидирующей рекомендации",49,0.070809,7.1
3,competitor_problem_solution_better,Конкурент сильнее в problem-solution,0,0.000000,0.0


In [18]:
gap_by_category = (
    df.groupby("категория")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_category

,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better
категория,,,,,
baby care,59.8,17.6,25.5,0.0,18.6
hair care,67.3,4.5,23.6,0.0,6.4
oral care,65.0,1.5,22.2,0.0,3.9
ОШИБКА,0.0,0.0,0.0,0.0,0.0


In [19]:
gap_by_query_type = (
    df.groupby("тип_запроса")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_query_type

,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better
тип_запроса,,,,,
ОШИБКА,0.0,0.0,0.0,0.0,0.0
брендовый,18.0,4.9,31.1,0.0,3.4
категорийный,82.5,12.0,35.5,0.0,13.4
проблемно-решающий,91.2,6.4,3.9,0.0,11.8


In [20]:
domains = df[["запрос", "категория", "тип_запроса", "домены_источников"] + gap_cols].copy()

domains["домены_источников"] = (
    domains["домены_источников"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.split(",")
)

domains = domains.explode("домены_источников")
domains["домен"] = domains["домены_источников"].str.strip()
domains = domains[domains["домен"] != ""].copy()

platform_map = {
    "ozon.ru": "маркетплейс",
    "wildberries.ru": "маркетплейс",
    "market.yandex.ru": "маркетплейс",
    "apteka.ru": "аптека/e-com",
    "goldapple.ru": "beauty retail",
    "kp.ru": "медиа/обзоры",
    "vc.ru": "медиа/статьи",
}

domains["platform_type"] = domains["домен"].map(platform_map).fillna("other")

In [21]:
domains

,запрос,категория,тип_запроса,домены_источников,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better,домен,platform_type
0,herbal essences алоэ,hair care,брендовый,poryadok.ru,0,0,0,0,0,poryadok.ru,other
0,herbal essences алоэ,hair care,брендовый,ozon.ru,0,0,0,0,0,ozon.ru,маркетплейс
0,herbal essences алоэ,hair care,брендовый,global.podrygka.ru,0,0,0,0,0,global.podrygka.ru,other
0,herbal essences алоэ,hair care,брендовый,goldapple.ru,0,0,0,0,0,goldapple.ru,beauty retail
0,herbal essences алоэ,hair care,брендовый,online.globus.ru,0,0,0,0,0,online.globus.ru,other
...,...,...,...,...,...,...,...,...,...,...,...
691,маска бальзам шампунь для волос,hair care,категорийный,letu.ru,1,0,0,0,0,letu.ru,other
691,маска бальзам шампунь для волос,hair care,категорийный,ozon.ru,1,0,0,0,0,ozon.ru,маркетплейс
691,маска бальзам шампунь для волос,hair care,категорийный,biofollica.ru,1,0,0,0,0,biofollica.ru,other
691,маска бальзам шампунь для волос,hair care,категорийный,wildberries.ru,1,0,0,0,0,wildberries.ru,маркетплейс


In [22]:
top_domains = domains["домен"].value_counts().head(15).index

gap_by_domain = (
    domains[domains["домен"].isin(top_domains)]
    .groupby("домен")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_domain

,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better
домен,,,,,
apteka.ru,52.7,3.1,26.4,0.0,0.0
detmir.ru,46.4,13.0,37.7,0.0,13.0
doctorslon.ru,48.7,2.6,32.9,0.0,1.3
dzen.ru,76.9,6.4,7.7,0.0,16.7
goldapple.ru,48.1,3.8,30.8,0.0,9.6
gorzdrav.org,56.9,10.3,37.9,0.0,5.2
kp.ru,78.2,19.2,5.1,0.0,35.9
market.yandex.ru,47.9,8.0,38.8,0.0,4.9
megapteka.ru,59.6,10.5,17.5,0.0,12.3


In [23]:
gap_by_platform_type = (
    domains.groupby("platform_type")[gap_cols]
    .mean()
    .mul(100)
    .round(1)
)

gap_by_platform_type

,no_pg_in_ai,pg_not_leading,no_selection_criteria,competitor_problem_solution_better,competitor_comparison_better
platform_type,,,,,
beauty retail,48.1,3.8,30.8,0.0,9.6
other,65.7,7.5,19.2,0.0,10.8
аптека/e-com,52.7,3.1,26.4,0.0,0.0
маркетплейс,51.4,10.4,42.1,0.0,5.3
медиа/обзоры,78.2,19.2,5.1,0.0,35.9
медиа/статьи,59.8,29.3,9.8,0.0,53.3


In [24]:
### САМОЕ НА МОЙ ВЗГЛЯД ВАЖНОЕ - для "проигрышных запросов". только по кейсам, где выигрывают конкуренты
losing_queries = df[
    df["кто_выигрывает"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("конкуренты")
].copy()

gap_cols = list(GAP_PATTERNS.keys())

losing_gap_summary = (
    losing_queries[gap_cols]
    .mean()
    .mul(100)
    .round(1)
    .rename("gap_rate_pct")
    .reset_index()
    .rename(columns={"index": "gap"})
)

losing_gap_summary["gap_label"] = losing_gap_summary["gap"].map(GAP_LABELS)

losing_gap_summary.sort_values("gap_rate_pct", ascending=False)

,gap,gap_rate_pct,gap_label
0,no_pg_in_ai,90.8,Нет присутствия P&G в AI-ответе
2,no_selection_criteria,20.1,Нет объяснения критериев выбора
4,competitor_comparison_better,13.7,Конкурент сильнее в comparison
1,pg_not_leading,9.2,"P&G есть, но не в лидирующей рекомендации"
3,competitor_problem_solution_better,0.0,Конкурент сильнее в problem-solution
